In [11]:
import numpy as np
import xarray as xr

In [12]:
#Nakata dataset
ice_prod_series = xr.open_dataset("Datasets/ice_prod_5km_Nakata.nc") #complete series month by month

#Burgard T-S profiles
T_S_Burgard = xr.open_dataset("Datasets/T_S_mean_prof_corrected_km_contshelf_and_offshore_1980-2018_oneFRIS.nc")


# Masks for defining ice shelves and front lines
IS_mask_Burgard = xr.open_dataset("Datasets/nemo_5km_isf_masks_and_info_and_distance_new_oneFRIS.nc")

# Box geometries
Box_1D_Burgard = xr.open_dataset("Datasets/nemo_5km_boxes_1D_oneFRIS.nc")
Box_2D_Burgard = xr.open_dataset("Datasets/nemo_5km_boxes_2D_oneFRIS.nc")

# Grid area
Grid_area = xr.open_dataset("Datasets/gridarea.nc")

#bed machine
Bedmachine = xr.open_dataset("Datasets/bedmachine_5km.nc")

In [13]:
# Let define isf of interest

index_isf = np.array([10,11,31,52,66,43,45,48,33])
def index2name(N):
    return IS_mask_Burgard.isf_name.sel(Nisf = N).values

name_isf =[]
for N in index_isf:
    name_isf.append(index2name(N))
name_isf = np.array(name_isf)
# ['Ross', 'Filchner-Ronne', 'Amery', 'Dotson', 'Pine Island', 'Thwaites', 'Riiser-Larsen', 'Roi Baudouin', 'Totten']

box_number = []
for N in index_isf:
    if N==52: # want to set nbox=2 for Dotson
        box_number.append(2)
    else:
        box_number.append(Box_1D_Burgard.nD_config.sel(Nisf=N).values[-1])
box_number = np.array(box_number)
# [5, 5, 3, 2, 2, 3, 3, 2]

Ac_list = []
frac_list = []
depth_list = []
Vc_list = []
for index, name, nbox in zip(index_isf,name_isf,box_number):
    Ac = Box_1D_Burgard.box_area.sel(box_nb_tot=1).sel(box_nb=1).sel(Nisf=index).values
    Ac_list.append(Ac) # m²

    Vc = np.nansum(((Bedmachine.surface-Bedmachine.thickness-Bedmachine.bed)*Grid_area.cell_area).where(IS_mask_Burgard.ISF_mask==index).values) #m^3
    Vc_list.append(Vc)
    
    print(name+' :',str(np.round(Ac/1e6/1e3,2))+' e3 km²     (Vc = '+str(np.round(Vc/1e12,1))+'e12 m^3)')
    print('#  ','frac','  <h> (m)')
    
    fracs = np.zeros(5)
    depths = np.zeros(5)
    for i in range(1,nbox+1): #weird range because of python norms..
        Aci = Box_1D_Burgard.box_area.sel(Nisf=index).sel(box_nb_tot=nbox).sel(box_nb=i).values
        frac = Aci/Ac
        fracs[i-1] = frac
        depth = Box_1D_Burgard.box_depth_below_surface.sel(Nisf=index).sel(box_nb_tot=nbox).sel(box_nb=i).values
        depths[i-1] = depth
        print(str(i)+' :', np.round(frac,2),'  ',str(np.round(depth,0)))
    print(' ')
    frac_list.append(fracs)
    depth_list.append(depths)

Ac_list = np.array(Ac_list)
frac_list = np.array(frac_list)
depth_list = -np.array(depth_list)
Vc_list = np.array(Vc_list)

Ross : 469.28 e3 km²     (Vc = 127.8e12 m^3)
#   frac   <h> (m)
1 : 0.21    -474.0
2 : 0.17    -400.0
3 : 0.19    -360.0
4 : 0.19    -334.0
5 : 0.24    -293.0
 
Filchner-Ronne : 416.61 e3 km²     (Vc = 134.8e12 m^3)
#   frac   <h> (m)
1 : 0.28    -870.0
2 : 0.17    -732.0
3 : 0.16    -564.0
4 : 0.17    -460.0
5 : 0.22    -352.0
 
Amery : 58.83 e3 km²     (Vc = 26.8e12 m^3)
#   frac   <h> (m)
1 : 0.54    -746.0
2 : 0.23    -440.0
3 : 0.23    -266.0
 
Dotson : 4.94 e3 km²     (Vc = 2.2e12 m^3)
#   frac   <h> (m)
1 : 0.4    -426.0
2 : 0.6    -320.0
 
Pine Island : 5.73 e3 km²     (Vc = 1.7e12 m^3)
#   frac   <h> (m)
1 : 0.45    -431.0
2 : 0.55    -331.0
 
Riiser-Larsen : 41.82 e3 km²     (Vc = 5.8e12 m^3)
#   frac   <h> (m)
1 : 0.2    -350.0
2 : 0.25    -256.0
3 : 0.54    -216.0
 
Roi Baudouin : 32.98 e3 km²     (Vc = 4.1e12 m^3)
#   frac   <h> (m)
1 : 0.2    -297.0
2 : 0.28    -251.0
3 : 0.52    -218.0
 
Totten : 6.58 e3 km²     (Vc = 3.2e12 m^3)
#   frac   <h> (m)
1 : 0.56    -927.0
2 :

In [14]:
from scipy.spatial import cKDTree

### Clara Burgard script
# find shortest distance of isf_points to line
def distance_isf_points_from_line(whole_domain,isf_points_da,line_points_da):

    """
    Compute the distance between ice shelf points and a line.

    This function computes the distance between ice shelf points and a line. This line can be the grounding
    line or the ice shelf front.

    Parameters
    ----------
    whole_domain : xarray.DataArray
        ice-shelf mask - all ice shelves are represented by a number, all other points (ocean, land) set to nan
    isf_points_da : xarray.DataArray
        array containing only points from one ice shelf
    line_points_da : xarray.DataArray
        mask representing the grounding line or ice shelf front mask corresponding to the ice shelf selected in ``isf_points_da``

    Returns
    -------
    xr_dist_to_line : xarray.DataArray
        distance of the each ice shelf point to the given line of interest
    """

    # add a common dimension 'grid' along which to stack
    stacked_isf_points = isf_points_da.stack(grid=['y', 'x'])
    stacked_line = line_points_da.stack(grid=['y', 'x'])

    # remove nans
    filtered_isf_points = stacked_isf_points[stacked_isf_points>0]
    filtered_line = stacked_line[stacked_line>0]

    # write out the y,x pairs behind the dimension 'grid'
    grid_isf_points = filtered_isf_points.indexes['grid'].to_frame().values.astype(float)
    grid_line = filtered_line.indexes['grid'].to_frame().values.astype(float)

    # create tree to line and compute distance
    tree_line = cKDTree(grid_line)
    dist_yx_to_line, _ = tree_line.query(grid_isf_points)

    # add the coordinates of the previous variables
    xr_dist_to_line = filtered_isf_points.copy(data=dist_yx_to_line)
    # put 1D array back into the format of the grid and put away the 'grid' dimension
    xr_dist_to_line = xr_dist_to_line.unstack('grid')
    # choose to reindex on initial grid (whole domain)
    xr_dist_to_line = xr_dist_to_line.reindex_like(whole_domain)

    return xr_dist_to_line

#if you want to apply an automatic mask based on the distance from the front of the ice shelf
def auto_mask(index,ice_prod_data, dIF_max, ice_prod_min):
    whole_domain = IS_mask_Burgard.ISF_mask
    isf_points_da = IS_mask_Burgard.ISF_mask
    line_points_da = IS_mask_Burgard.IF_mask.where(IS_mask_Burgard.IF_mask==index)
    dIF = distance_isf_points_from_line(whole_domain,isf_points_da,line_points_da)
    
    return (dIF.where(dIF<dIF_max)*ice_prod_data.where(ice_prod_data>ice_prod_min))>=0

#if you want to define tour geographical mask manually
def manual_mask(ice_prod_data,xmin,xmax,ymin,ymax,ice_prod_min):
    return ice_prod_data.where(ice_prod_data>ice_prod_min).where(IS_mask_Burgard.x>xmin).where(IS_mask_Burgard.x<xmax).where(IS_mask_Burgard.y>ymin).where(IS_mask_Burgard.y<ymax)>=0

# return the area of the Polynya Ap and the sea ice formation for a certain mask at a certain time
def surface_forcing(time,mask):
    Ap_grid = Grid_area.cell_area*mask_bin
    Ap = np.sum(Ap_grid) #m²
    Omega_avg = np.sum(ice_prod_data*mask_bin*Ap_grid)/Ap #m/30d

    if np.isnan(Omega_avg): #management of the case where there is no polynya
        Omega_avg = 0
    
    return Ap, Omega_avg
    

### First test with Ross ice shelf

In [15]:
time = '2003-03-01T00:00:00.000000000' #selected month
index = 10
ice_prod_data = ice_prod_series["prod"].sel(time=time)
name = 'Ross'
dIF_max = 150e3 #m  # maximal distance from the ice shelf front
ice_prod_min = 0.5 #m/30d  # minimal ice formation rate to consider the cell as a polynya

mask_bin = auto_mask(index,ice_prod_data, dIF_max, ice_prod_min)

Ap, Omega_avg = surface_forcing(time,mask_bin)

if np.isnan(Omega_avg): #management of the case where there is no polynya
    Omega_avg = 0

print('Ap : ',np.round(Ap.values/1e9,2),'e3 km²')
print('<Omega> : ', np.round(Omega_avg.values,2),'m/30d')

Ap :  35.15 e3 km²
<Omega> :  0.94 m/30d


### Compute monthly surface forcing for each ice shelf

In [16]:
# can take some minutes
ice_prod_min = 0.5 # m/30d threshold to consider polynya cells
auto_list = [True,True,True,False,False,True,True,False,False] # is the mask automatically set ?
dIF_max_list = [150e3,100e3,75e3,0,0,35e3,35e3,0,0] # m max distance from the front of the polynya (in auto)
coords_list = [0,0,0,(-1800000,-1500000,-730000,-600000),(-1700000,-1580000,-400000,-300000),0,0,(21.5e5,23.3e5,-13.75e5,-10e5),(21.3e5,22.1e5,-14.75e5,-13.75e5)] # m limit coordinates of the manual mask

Ap_tab = np.zeros((len(index_isf),len(ice_prod_series.time.values)))
Omega_tab = np.zeros((len(index_isf),len(ice_prod_series.time.values)))

for i, index, auto, dIF_max, coords in zip(np.arange(len(index_isf)),index_isf,auto_list,dIF_max_list,coords_list):
    print(index)
    for j, time in enumerate(ice_prod_series.time.values):
        ice_prod_data = ice_prod_series["prod"].sel(time=time)
        if auto:
            mask_bin = auto_mask(index,ice_prod_data, dIF_max, ice_prod_min)
        else:
            xmin,xmax,ymin,ymax = coords
            mask_bin =  manual_mask(ice_prod_data,xmin,xmax,ymin,ymax,ice_prod_min)

        Ap, Omega_avg = surface_forcing(time,mask_bin)
        Ap_tab[i,j] = Ap #m²
        Omega_tab[i,j] = Omega_avg #m/30d

10
11
31
52
66
43
45
48
33


### Compute WDW properties $T_0$ and $S_0$ and surface water properties $T_{surf}$ and $S_{surf}$ 

In [17]:
# First test with Ross ice shelf
time = 2003 #selected year
index = 10
name = 'Ross'
offshore = True
T_S_dataset = T_S_Burgard.sel(Nisf=index).sel(time=time)

if offshore:
    domain = 1000
else:
    domain = 100 # Continental shelf profiles are taken 100 km from the ice shelf front

d_avg = IS_mask_Burgard.sel(Nisf=index).front_bot_depth_avg.values
T0 = T_S_dataset.sel(profile_domain=domain).theta_ocean.sel(depth=d_avg,method="nearest").values
S0 = T_S_dataset.sel(profile_domain=domain).salinity_ocean.sel(depth=d_avg,method="nearest").values

print('T0 = '+str(np.round(T0,2))+' °C', ' S0 = '+str(np.round(S0,2))+' PSU')

# surface water propertie offshore
T_surf = T_S_dataset.sel(profile_domain=1000).theta_ocean.sel(depth=0,method="nearest").values
S_surf = T_S_dataset.sel(profile_domain=1000).salinity_ocean.sel(depth=0,method="nearest").values

print('T_surf = '+str(np.round(T_surf,2))+' °C', ' S_surf = '+str(np.round(S_surf,2))+' PSU')

T0 = 1.74 °C  S0 = 34.69 PSU
T_surf = -1.71 °C  S_surf = 33.84 PSU


In [18]:
# function to compute water properties
def water_properties(offshore,T_S_dataset):
    if offshore:
        domain = 1000
    else:
        domain = 100 # Continental shelf profiles are taken 100 km from the ice shelf front
    
    d_avg = IS_mask_Burgard.sel(Nisf=index).front_bot_depth_avg.values
    T0 = T_S_dataset.sel(profile_domain=domain).theta_ocean.sel(depth=d_avg,method="nearest").values
    S0 = T_S_dataset.sel(profile_domain=domain).salinity_ocean.sel(depth=d_avg,method="nearest").values
    
    T_surf = T_S_dataset.sel(profile_domain=1000).theta_ocean.sel(depth=0,method="nearest").values
    S_surf = T_S_dataset.sel(profile_domain=1000).salinity_ocean.sel(depth=0,method="nearest").values

    return T0, S0, T_surf, S_surf

In [19]:
# compute WDW properties T0 and S0 and surface water properties T_surf and S_surf for each ice shelf

# offshore_list = 3*[True]+6*[False] 
offshore_list = 9*[True] # Do we take WDW properties offshore ?

T0_tab = np.zeros((len(index_isf),8))
S0_tab = np.zeros((len(index_isf),8))
T_surf_tab = np.zeros((len(index_isf),8))
S_surf_tab = np.zeros((len(index_isf),8))

for i, index, offshore in zip(np.arange(len(index_isf)),index_isf,offshore_list):
    for j, time in enumerate(np.arange(2003,2011)):
        T_S_dataset = T_S_Burgard.sel(Nisf=index).sel(time=time)
        T0, S0, T_surf, S_surf = water_properties(offshore,T_S_dataset)

        T0_tab[i,j] = T0
        S0_tab[i,j] = S0
        T_surf_tab[i,j] = T_surf
        S_surf_tab[i,j] = S_surf

### Creating dataset that contains inputs parameters for the low-dimensionnal model

In [20]:
Dataset= xr.Dataset(
    {
        "name_isf": (["Nisf"],name_isf,{"descr":"Name of the ice shelves"}),
        "Nbox": (["Nisf"],box_number,{"descr":"Number of cavity boxes"}),
        "Ac": (["Nisf"],Ac_list,{"descr":"Ice ocean interface area [m^2]"}),
        "depth": (["Nisf","box"], depth_list, {"descr":"Average depth of the cavity boxes [m]"}),
        "Vc": (["Nisf"],Vc_list,{"descr":"Ice shelf volume [m^3]"}),
        "frac": (["Nisf","box"],frac_list,{"descr":"Area fractions of the ice ocean interface"}),
        "Ap": (["Nisf","time"],Ap_tab,{"descr":"Polynya area [m^2] (full series of the ice production averaged)"}),
        "Omega": (["Nisf","time"],Omega_tab,{"descr":"Sea ice formation rate [m/30d] (full series of the ice production averaged)"}),
        "T0": (["Nisf","time_year"],T0_tab,{"descr":"WDW temperature (100km from the frontline) [°C]"}),
        "S0": (["Nisf","time_year"],S0_tab,{"descr":"WDW salinity (100km from the frontline) [PSU]"}),
        "T_surf": (["Nisf","time_year"],T_surf_tab,{"descr":"Surface temperature (model input) [°C]"}),
        "S_surf": (["Nisf","time_year"],S_surf_tab,{"descr":"Surface salinity (model input) [PSU]"}),
        
    },
    coords={
        "Nisf": index_isf,
        "box": np.arange(1,5+1),
        "time": ice_prod_series.time,
        "time_year": np.arange(2003,2011),
    },
    attrs={
        "Global": "Dataset containing the input parameters for the box model",
        "Nisf": "Index of the ice shelves (from Burgard 2022)",
        "box": "box number (but not ntotal box number)",
    }
)

In [ ]:
#Dataset.to_netcdf("model_input_dataset.nc")

In [24]:
Dataset.T0.mean(dim="time_year")

<xarray.DataArray 'T0' (Nisf: 9)> Size: 72B
array([1.78239458, 0.0157305 , 0.61827091, 1.79844149, 1.94903731,
       0.60326403, 0.54436571, 1.07774624, 1.1648266 ])
Coordinates:
  * Nisf     (Nisf) int64 72B 10 11 31 52 66 43 45 48 33